# Exercices — Syntaxe pandas

Cinq exercices progressifs sur le dataset des accidents de la route.
Chaque exercice mobilise un pan du cours : sélection, filtre, groupby, merge, strings.

> **Consigne** : écrivez votre réponse dans la cellule vide fournie.
> La solution est cachée dans le bloc `<details>` — consultez-la après avoir essayé.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_accidents_usagers

df = load_accidents()          # caractéristiques + lieux (left join)
usagers = load_accidents_usagers()

print("df      :", df.shape)
print("usagers :", usagers.shape)

***
## Exercice 1 — Sélection de colonnes

À partir de `df`, créez un nouveau DataFrame `df_reduit` qui ne contient que les colonnes suivantes :
`Num_Acc`, `dep`, `mois`, `lum`, `surf`, `vma`.

Puis affichez :
- sa forme (`.shape`)
- les 5 premières lignes
- la liste de ses types (`.dtypes`)

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
cols = ["Num_Acc", "dep", "mois", "lum", "surf", "vma"]
df_reduit = df[cols]

print(df_reduit.shape)
display(df_reduit.head())
print(df_reduit.dtypes)
```

**Points à retenir :**
- `df[liste_colonnes]` retourne un DataFrame (double crochet).
- `dep` est de type `str` — attention à ne pas le traiter comme un entier.
- `vma` (vitesse maximale autorisée) peut contenir des 0 ou valeurs aberrantes à explorer.

</details>

***
## Exercice 2 — Filtrage

Combien d'accidents ont eu lieu **à Paris** (`dep == '75'`) **en janvier** (`mois == 1`),
**par temps de nuit sans éclairage** (`lum == 4`) ?

1. Utilisez `.query()` pour filtrer.
2. Affichez le nombre d'accidents trouvés.
3. **Bonus** : quelle est la répartition par `col` (type de collision) dans ce sous-ensemble ?
   Utilisez `.value_counts()`.

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
# Filtre
filtre = df.query("dep == '75' and mois == 1 and lum == 4")
print(f"{len(filtre)} accidents trouvés")

# Bonus : répartition par type de collision
filtre["col"].value_counts()
```

**Points à retenir :**
- `dep == '75'` : guillemets obligatoires, `dep` est une chaîne de caractères.
- Dans `.query()`, les strings doivent être entourées de guillemets simples si la chaîne de `.query()` utilise des guillemets doubles, et vice versa.
- `lum == 4` : nuit sans éclairage public (valeur codifiée — voir `_data.py` pour le détail).

</details>

***
## Exercice 3 — Groupby

Pour chaque département (`dep`), calculez :
- le **nombre total d'accidents**
- la **vitesse maximale autorisée médiane** (`vma`)

Renommez les colonnes résultat `n_accidents` et `vma_mediane`.
Affichez les **10 départements les plus accidentogènes** (nombre d'accidents décroissant).

> **Indice** : utilisez `.agg()` avec des noms explicites (named aggregations).

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
(
    df
    .groupby("dep")
    .agg(
        n_accidents=("Num_Acc", "count"),
        vma_mediane=("vma", "median"),
    )
    .sort_values("n_accidents", ascending=False)
    .head(10)
)
```

**Points à retenir :**
- La syntaxe `col_résultat=("col_source", "fonction")` dans `.agg()` nomme directement les colonnes de sortie.
- `.sort_values(ascending=False)` trie du plus grand au plus petit.
- Observez que Paris (75) n'est pas nécessairement en tête — les départements à fort réseau routier (13, 69, 59…) apparaissent souvent en premier.

</details>

***
## Exercice 4 — Merge

Le fichier `usagers` contient une colonne `grav` (gravité de la blessure) :
- 1 = indemne
- 2 = tué
- 3 = blessé hospitalisé
- 4 = blessé léger

**Objectif** : trouver les 10 départements avec le plus grand nombre de **tués** (`grav == 2`).

Étapes suggérées :
1. Filtrer `usagers` pour ne garder que les tués.
2. Merger avec `df` sur `Num_Acc` pour récupérer le département.
3. Compter par `dep` et trier.

> **Indice** : un accident peut avoir plusieurs usagers — le merge peut produire plusieurs lignes par accident.

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
tues = usagers.query("grav == 2")

(
    tues
    .merge(df[["Num_Acc", "dep"]], on="Num_Acc", how="left")
    .groupby("dep")
    .agg(n_tues=("Num_Acc", "count"))
    .sort_values("n_tues", ascending=False)
    .head(10)
)
```

**Points à retenir :**
- On filtre `usagers` *avant* le merge pour réduire le volume de données.
- On ne récupère que les colonnes nécessaires de `df` : `df[["Num_Acc", "dep"]]`.
- Un tué = une ligne dans `usagers` avec `grav == 2`.
  `.count()` sur `Num_Acc` compte le nombre de lignes, c'est-à-dire le nombre de tués.

</details>

***
## Exercice 5 — Manipulation de chaînes

La colonne `hrmn` contient l'heure de l'accident au format `"HH:MM"` (ex : `"07:40"`).

1. Extrayez les **heures** comme des entiers (colonne `heure`).
2. Calculez le nombre d'accidents **par heure de la journée** (0 à 23).
3. Identifiez l'**heure la plus dangereuse** (le plus d'accidents).
4. **Bonus** : tracez un simple graphique en barres avec `.plot.bar()`.

> **Indice** : `hrmn` est une chaîne — utilisez `.str[:2]` ou `.str.split()` pour extraire les heures.

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
par_heure = (
    df
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))
    .groupby("heure")
    .agg(n_accidents=("Num_Acc", "count"))
)

heure_pic = par_heure["n_accidents"].idxmax()
print(f"Heure la plus dangereuse : {heure_pic}h ({par_heure.loc[heure_pic, 'n_accidents']} accidents)")

# Bonus
par_heure.plot.bar(y="n_accidents", figsize=(12, 4), legend=False, title="Accidents par heure")
```

**Points à retenir :**
- `.str[:2]` découpe les 2 premiers caractères de chaque chaîne — vectorisé, pas de boucle.
- `.astype(int)` convertit la sous-chaîne en entier.
- `.idxmax()` retourne l'**index** de la valeur maximale (l'heure, pas le nombre d'accidents).
- On peut tout enchaîner en un seul pipeline avec `.assign()` + `.groupby()` + `.agg()`.

</details>